In [1]:
!pip install rdflib

In [58]:
import numpy as np
import rdflib
from rdflib import Graph, Literal, URIRef
from rdflib.namespace import RDF, XSD
from datetime import datetime, timedelta
import random

# Define namespaces
ex = rdflib.Namespace("http://example.org/")
health = rdflib.Namespace("http://example.org/health/")
finance = rdflib.Namespace("http://example.org/finance/")
general = rdflib.Namespace("http://example.org/general/")
time = rdflib.Namespace("http://example.org/time/")

# Create an empty graph
g = Graph()

# Bind namespaces to prefixes for better readability in the output
g.bind("ex", ex)
g.bind("health", health)
g.bind("finance", finance)
g.bind("general", general)
g.bind("time", time)

# Set the start date for our simulation
start_date = datetime(2024, 10, 20, 10, 0, 0) # October 20, 2024, 10:00 AM IST
num_months = 60
current_date = start_date

# Lists of entities and relations for each domain
persons = [ex["JohnDoe"], ex["JaneSmith"], ex["Robert"], ex["Alice"]]
diseases = [health["Flu"], health["COVID19"], health["Allergy"]]
symptoms = [health["Fever"], health["Cough"], health["SoreThroat"], health["Fatigue"], health["RunnyNose"]]
medications = [health["Paracetamol"], health["Ibuprofen"], health["Antihistamine"]]
doctors = [health["DrAlice"], health["DrBob"]]
hospitals = [health["CityHospital"], health["GeneralClinic"]]

accounts = [finance["JohnChecking"], finance["JaneSavings"], finance["RobertCredit"]]
transactions = [finance["T1"], finance["T2"], finance["T3"], finance["T4"], finance["T5"]]
financial_products = [finance["StockA"], finance["BondB"], finance["MutualFundC"]]
banks = [finance["CityBank"], finance["NationalCreditUnion"]]

locations = [general["Home"], general["Office"], general["Gym"], general["Restaurant"]]
activities = [general["Meeting"], general["Workout"], general["Lunch"], general["Travel"]]
items = [general["BookA"], general["Laptop"], general["Phone"]]

relations_health = [health["hasSymptom"], health["diagnosedWith"], health["prescribed"], health["visitedDoctor"], health["atHospital"]]
relations_finance = [finance["hasAccount"], finance["madeTransaction"], finance["investedIn"], finance["accountAt"], finance["transactionAmount"]]
relations_general = [general["locatedAt"], general["participatedIn"], general["ownsItem"]]

# Function to add a temporal triple
def add_temporal_triple(subject, predicate, object, timestamp):
    g.add((subject, predicate, object))
    g.add((subject, time["timestamp"], Literal(timestamp.isoformat(), datatype=XSD.dateTime)))

# Simulate events over 6 months
for month in range(num_months):
    days_in_month = (current_date.replace(month=current_date.month % 12 + 1, day=1) - timedelta(days=1)).day
    for day in range(days_in_month):
        for hour in range(random.randint(1, 5)): # Simulate a few events per day
            minute = random.randint(0, 59)
            second = random.randint(0, 59)
            event_time = current_date.replace(day=day + 1, hour=random.randint(8, 18), minute=minute, second=second)

            # Simulate Health events
            if random.random() < 0.15:
                person = random.choice(persons)
                if random.random() < 0.4:
                    disease_or_symptom = random.choice(diseases + symptoms)
                    relation = random.choice([health["hasSymptom"], health["diagnosedWith"]])
                    add_temporal_triple(person, relation, disease_or_symptom, event_time)
                elif random.random() < 0.3:
                    person = random.choice(persons)
                    med = random.choice(medications)
                    doctor = random.choice(doctors)
                    add_temporal_triple(doctor, health["prescribed"], med, event_time)
                    add_temporal_triple(person, health["prescribed"], med, event_time)
                    add_temporal_triple(person, health["visitedDoctor"], doctor, event_time)
                elif random.random() < 0.2:
                    person = random.choice(persons)
                    hospital = random.choice(hospitals)
                    add_temporal_triple(person, health["atHospital"], hospital, event_time)

            # Simulate Finance events
            if random.random() < 0.2:
                person = random.choice(persons)
                account = random.choice(accounts)
                relation_fin = random.choice(relations_finance[:-1]) # Exclude transactionAmount initially
                add_temporal_triple(person, finance["hasAccount"], account, event_time)
                if random.random() < 0.3:
                    transaction = random.choice(transactions)
                    add_temporal_triple(account, finance["madeTransaction"], transaction, event_time)
                    amount = round(random.uniform(10, 1000), 2)
                    add_temporal_triple(transaction, finance["transactionAmount"], Literal(amount, datatype=XSD.float), event_time)
                elif random.random() < 0.1:
                    product = random.choice(financial_products)
                    add_temporal_triple(person, finance["investedIn"], product, event_time)
                elif random.random() < 0.2:
                    bank = random.choice(banks)
                    add_temporal_triple(account, finance["accountAt"], bank, event_time)

            # Simulate General events
            if random.random() < 0.3:
                person = random.choice(persons)
                location = random.choice(locations)
                add_temporal_triple(person, general["locatedAt"], location, event_time)
            elif random.random() < 0.25:
                person = random.choice(persons)
                activity = random.choice(activities)
                add_temporal_triple(person, general["participatedIn"], activity, event_time)
            elif random.random() < 0.1:
                person = random.choice(persons)
                item = random.choice(items)
                add_temporal_triple(person, general["ownsItem"], item, event_time)

    current_date += timedelta(days=30) # Approximate month increment


In [59]:
g.serialize(destination="simulated_tkg.ttl", format="turtle")

<Graph identifier=Naff163a903094a0dbf17f9a5273569d2 (<class 'rdflib.graph.Graph'>)>

In [60]:
len(g)

5765

Subgraph extraction

In [61]:
!pip install torch pykeen


In [62]:
from rdflib import Graph, URIRef, Literal
from pykeen.pipeline import pipeline
import torch
from collections import defaultdict
import random

# Load the TKG
g = Graph()
g.parse("simulated_tkg.ttl", format="ttl")

# Extract snapshot at time t (you can also define a time window)
def extract_snapshot(graph, time_literal):
    snapshot = Graph()
    for s, p, o in graph:
        if "timestamp" in str(p) and str(o) == time_literal:
            snapshot.add((s, p, o))
            for triple in graph.triples((s, None, None)):
                snapshot.add(triple)
    return snapshot


In [63]:
def rdf_to_triples(graph):
    triples = []
    for s, p, o in graph:
        if isinstance(o, URIRef):
            triples.append((str(s), str(p), str(o)))
    return triples


# Build namespace prefix map
prefix_map = {str(ns): prefix for prefix, ns in g.namespace_manager.namespaces()}

# Global label shortener
def prefixed_label(uri):
    uri_str = str(uri)
    for ns_uri, prefix in prefix_map.items():
        if uri_str.startswith(ns_uri):
            return f"{prefix}:{uri_str[len(ns_uri):]}"
    return uri_str  # fallback to full URI if no match



In [64]:
#### Candidate note/triple selection

def identify_candidates(triples, top_k=5):
    degree_count = defaultdict(int)
    for s, _, o in triples:
        degree_count[s] += 1
        degree_count[o] += 1
    top_nodes = sorted(degree_count, key=degree_count.get, reverse=True)[:top_k]
    return top_nodes


In [65]:
!pip uninstall -y numpy gensim
!pip install numpy==1.23.5 gensim --force-reinstall
!pip install node2vec

Found existing installation: numpy 1.26.4
Uninstalling numpy-1.26.4:
  Successfully uninstalled numpy-1.26.4
Found existing installation: gensim 4.3.3
Uninstalling gensim-4.3.3:
  Successfully uninstalled gensim-4.3.3
  Using cached numpy-1.23.5-cp311-cp311-manylinux_2_17_x86_64.manylinux2014_x86_64.whl.metadata (2.3 kB)
  Using cached gensim-4.3.3-cp311-cp311-manylinux_2_17_x86_64.manylinux2014_x86_64.whl.metadata (8.1 kB)
  Using cached scipy-1.13.1-cp311-cp311-manylinux_2_17_x86_64.manylinux2014_x86_64.whl.metadata (60 kB)
  Using cached smart_open-7.1.0-py3-none-any.whl.metadata (24 kB)
  Using cached wrapt-1.17.2-cp311-cp311-manylinux_2_5_x86_64.manylinux1_x86_64.manylinux_2_17_x86_64.manylinux2014_x86_64.whl.metadata (6.4 kB)
Using cached numpy-1.23.5-cp311-cp311-manylinux_2_17_x86_64.manylinux2014_x86_64.whl (17.1 MB)
Using cached gensim-4.3.3-cp311-cp311-manylinux_2_17_x86_64.manylinux2014_x86_64.whl (26.7 MB)
Using cached scipy-1.13.1-cp311-cp311-manylinux_2_17_x86_64.manylinu

  Using cached numpy-1.26.4-cp311-cp311-manylinux_2_17_x86_64.manylinux2014_x86_64.whl.metadata (61 kB)
Using cached numpy-1.26.4-cp311-cp311-manylinux_2_17_x86_64.manylinux2014_x86_64.whl (18.3 MB)
  Attempting uninstall: numpy
    Found existing installation: numpy 1.23.5
    Uninstalling numpy-1.23.5:
      Successfully uninstalled numpy-1.23.5
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
thinc 8.3.6 requires numpy<3.0.0,>=2.0.0, but you have numpy 1.26.4 which is incompatible.


In [66]:
### subgraph expansion

import networkx as nx
from node2vec import Node2Vec
from rdflib import URIRef

def triples_to_nx_graph(triples):
    G = nx.DiGraph()
    for h, r, t in triples:
        G.add_edge(str(h), str(t), label=str(r))
    return G

def get_embeddings_and_expand_simple(triples, candidate_nodes, max_depth=2, threshold=0.1):
    # Convert RDF triples to a NetworkX graph
    G = triples_to_nx_graph(triples)

    # Generate embeddings using Node2Vec (no training, unsupervised)
    node2vec = Node2Vec(G, dimensions=64, walk_length=10, num_walks=50, workers=1)
    model = node2vec.fit()

    # Normalize embeddings
    def similarity(n1, n2):
        try:
            return model.wv.similarity(str(n1), str(n2))
        except KeyError:
            return 0.0

    subgraphs = []
    for root in candidate_nodes:
        visited = set([root])
        frontier = [root]
        subgraph = []

        for _ in range(max_depth):
            next_frontier = []
            for h, r, t in triples:
                h_str, t_str = str(h), str(t)
                if h_str in frontier:
                    score = similarity(root, t_str)
                    #print ('similarity score: ', score)
                    if score >= threshold:
                        subgraph.append((h, r, t))
                        if t_str not in visited:
                            next_frontier.append(t_str)
                            visited.add(t_str)
            frontier = next_frontier

        if subgraph:
            subgraphs.append(subgraph)

    return subgraphs


In [67]:
from sentence_transformers import SentenceTransformer
from sklearn.metrics.pairwise import cosine_similarity
from node2vec import Node2Vec
import networkx as nx

# Load MiniLM model globally (only once)
semantic_model = SentenceTransformer("all-MiniLM-L6-v2")

def triples_to_nx_graph(triples):
    G = nx.DiGraph()
    for h, r, t in triples:
        G.add_edge(str(h), str(t), label=str(r))
    return G

def get_embeddings_and_expand_with_semantics(triples, candidate_nodes, max_depth=2, threshold=0.5, semantic_threshold=0.6):
    # Step 1: Build networkx graph
    G = triples_to_nx_graph(triples)

    # Step 2: Node2Vec embeddings
    node2vec = Node2Vec(G, dimensions=64, walk_length=10, num_walks=50, workers=1)
    model = node2vec.fit()

    def similarity(n1, n2):
        try:
            return model.wv.similarity(str(n1), str(n2))
        except KeyError:
            return 0.0

    # Step 3: Semantic filtering
    def is_semantically_similar(relation_uri, core_embedding, threshold):
        rel_label = relation_uri.split("/")[-1]
        emb = semantic_model.encode(rel_label)
        sim = cosine_similarity([core_embedding], [emb])[0][0]
        return sim >= threshold

    subgraphs = []

    for root in candidate_nodes:
        visited = set([root])
        frontier = [root]
        subgraph = []

        # Core embedding based on most frequent predicate around root
        root_triples = [triple for triple in triples if str(triple[0]) == root]
        if not root_triples:
            continue
        top_rel = max(root_triples, key=lambda x: sum(x[1] in r for _, r, _ in root_triples))[1]
        top_rel_label = top_rel.split("/")[-1]
        core_embedding = semantic_model.encode(top_rel_label)

        for _ in range(max_depth):
            next_frontier = []
            for h, r, t in triples:
                h_str, t_str = str(h), str(t)
                if h_str in frontier:
                    struct_score = similarity(root, t_str)
                    if struct_score >= threshold:
                        if is_semantically_similar(r, core_embedding, semantic_threshold):
                            subgraph.append((h, r, t))
                            if t_str not in visited:
                                next_frontier.append(t_str)
                                visited.add(t_str)
            frontier = next_frontier

        if subgraph:
            subgraphs.append(subgraph)

    return subgraphs


In [82]:
from sentence_transformers import SentenceTransformer
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score
from sklearn.metrics.pairwise import cosine_similarity
from collections import defaultdict
import numpy as np
import networkx as nx
from node2vec import Node2Vec

semantic_model = SentenceTransformer('all-MiniLM-L6-v2')

def triples_to_nx_graph(triples):
    G = nx.DiGraph()
    for h, r, t in triples:
        G.add_edge(str(h), str(t), label=str(r))
    return G

def embed_predicates(triples):
    predicates = [r.split("/")[-1] for _, r, _ in triples]
    embeddings = semantic_model.encode(predicates)
    return embeddings

def get_entropy(embeddings):
    sim_matrix = cosine_similarity(embeddings)
    avg_sim = np.mean(sim_matrix, axis=1)
    diversity = 1 - avg_sim
    return np.mean(diversity)

def expand_and_cluster_aspects(triples, top_k=10, max_depth=2, struct_threshold=0.3, k_clusters=2):
    G = triples_to_nx_graph(triples)

    node2vec = Node2Vec(G, dimensions=64, walk_length=10, num_walks=50, workers=1)
    model = node2vec.fit()

    def similarity(n1, n2):
        try:
            return model.wv.similarity(str(n1), str(n2))
        except KeyError:
            return 0.0

    # --- Step 1: Smart Candidate Selection based on embedding diversity ---
    node_neighbors = defaultdict(list)
    for h, _, t in triples:
        node_neighbors[h].append(t)
        node_neighbors[t].append(h)

    diversity_scores = {}
    for node in node_neighbors:
        local_triples = [triple for triple in triples if str(triple[0]) == node or str(triple[2]) == node]
        if len(local_triples) < 3:
            continue
        emb = embed_predicates(local_triples)
        diversity_scores[node] = get_entropy(emb)

    top_nodes = sorted(diversity_scores, key=diversity_scores.get, reverse=True)[:top_k]

    # --- Step 2: For each top node, extract local subgraph ---
    all_aspect_subgraphs = []

    for root in top_nodes:
        visited = set([root])
        frontier = [root]
        local_triples = []

        for _ in range(max_depth):
            next_frontier = []
            for h, r, t in triples:
                h_str, t_str = str(h), str(t)
                if h_str in frontier:
                    if similarity(root, t_str) >= struct_threshold:
                        local_triples.append((h, r, t))
                        if t_str not in visited:
                            next_frontier.append(t_str)
                            visited.add(t_str)
            frontier = next_frontier

        if not local_triples:
            continue

        # --- Step 3: Embed and cluster predicates into latent aspects ---
        embeddings = embed_predicates(local_triples)

        if len(embeddings) < 2:
            continue  # Not enough for clustering

        # Optional: Auto-tune k using silhouette score
        best_score = -1
        best_k = 2
        for k in range(2, min(6, len(embeddings))):
            kmeans = KMeans(n_clusters=k, random_state=42)
            labels = kmeans.fit_predict(embeddings)
            score = silhouette_score(embeddings, labels)
            if score > best_score:
                best_score = score
                best_k = k

        kmeans = KMeans(n_clusters=best_k, random_state=42)
        labels = kmeans.fit_predict(embeddings)

        # --- Step 4: Split triples into aspect subgraphs ---
        clustered_subgraphs = defaultdict(list)
        for triple, label in zip(local_triples, labels):
            clustered_subgraphs[label].append(triple)

        all_aspect_subgraphs.extend(clustered_subgraphs.values())

    return all_aspect_subgraphs


In [68]:
from rdflib.namespace import XSD

# Function to extract all unique timestamp literals from the graph
def get_all_timestamps(graph):
    timestamps = set()
    for _, p, o in graph:
        if 'timestamp' in str(p) and isinstance(o, Literal) and o.datatype == XSD.dateTime:
            timestamps.add(str(o))
    return sorted(timestamps)


In [69]:
timestamps = get_all_timestamps(g)

timestamps = get_all_timestamps(g)
print("Available timestamps:", len(timestamps))
# for i, ts in enumerate(timestamps):
#     print(f"{i}. {ts}")

# # Choose one (manually or programmatically)
# index = 2  # Example: third snapshot
# t = timestamps[index]
# print(f"Using snapshot at: {t}")


# Example: pick the earliest timestamp
t = timestamps[150]  # or timestamps[-1] for latest, timestamps[len(timestamps)//2] for middle

print(f"Using snapshot at time: {t}")


Available timestamps: 3616
Using snapshot at time: 2024-12-09T11:20:30


In [73]:
for i in range(len(timestamps)):
  t = timestamps[i]
  snapshot = extract_snapshot(g, t)
  triples = rdf_to_triples(snapshot)
  if len(triples) > 100:
    print (i, len(triples))

49 127
188 119
444 122
494 119
1161 122
1515 126
1711 119
1729 126
1777 130
1876 126
2403 119
2571 122
2728 119
3070 129
3248 126
3558 126


In [93]:
t = timestamps[1777]
snapshot = extract_snapshot(g, t)
triples = rdf_to_triples(snapshot)
candidates = identify_candidates(triples, top_k=100)
# subgraphs = get_embeddings_and_expand_simple(triples, candidates)
# subgraphs = get_embeddings_and_expand_with_semantics(
#     triples,
#     candidates,
#     max_depth=10,
#     threshold=0.05,              # structural threshold
#     semantic_threshold=0.4      # semantic coherence threshold
# )
subgraphs = expand_and_cluster_aspects(
    triples,
    top_k=50,           # number of candidate roots
    max_depth=15,        # expansion depth
    struct_threshold=0.1,
    k_clusters=2        # optional fixed cluster count (auto-tunes inside)
)


# for idx, sg in enumerate(subgraphs):
#     print(f"\nSubgraph {idx+1}:")
#     for triple in sg:
#         print(triple)
for idx, sg in enumerate(subgraphs):
    print(f"\nSubgraph {idx+1}:")
    for triple in sg:
        print(tuple(prefixed_label(x) for x in triple))


Computing transition probabilities:   0%|          | 0/42 [00:00<?, ?it/s]

Generating walks (CPU: 1): 100%|██████████| 50/50 [00:00<00:00, 6272.33it/s]



Subgraph 1:
('ex:JohnDoe', 'ex:health/hasSymptom', 'ex:health/SoreThroat')
('ex:JohnDoe', 'ex:health/atHospital', 'ex:health/GeneralClinic')

Subgraph 2:
('ex:JohnDoe', 'ex:general/participatedIn', 'ex:general/Lunch')
('ex:JohnDoe', 'ex:health/diagnosedWith', 'ex:health/SoreThroat')

Subgraph 3:
('ex:JohnDoe', 'ex:general/locatedAt', 'ex:general/Office')
('ex:JohnDoe', 'ex:general/locatedAt', 'ex:general/Gym')

Subgraph 4:
('ex:JohnDoe', 'ex:health/visitedDoctor', 'ex:health/DrBob')
('ex:JohnDoe', 'ex:general/ownsItem', 'ex:general/BookA')

Subgraph 5:
('ex:JaneSmith', 'ex:general/locatedAt', 'ex:general/Office')

Subgraph 6:
('ex:JaneSmith', 'ex:general/participatedIn', 'ex:general/Lunch')
('ex:JaneSmith', 'ex:health/diagnosedWith', 'ex:health/SoreThroat')
('ex:JaneSmith', 'ex:health/hasSymptom', 'ex:health/SoreThroat')

Subgraph 7:
('ex:JaneSmith', 'ex:general/ownsItem', 'ex:general/Phone')
('ex:JaneSmith', 'ex:general/ownsItem', 'ex:general/Laptop')

Subgraph 8:
('ex:Alice', 'ex:he

In [71]:
len(triples)

39

TEmporal Evolutions

In [89]:
def match_subgraph_in_snapshot(subgraph, snapshot):
    snapshot_triples = set((str(s), str(p), str(o)) for s, p, o in snapshot if isinstance(o, URIRef))
    original = set((str(s), str(p), str(o)) for s, p, o in subgraph)

    overlap = original & snapshot_triples
    added = snapshot_triples - original
    removed = original - snapshot_triples

    return {
        "overlap_count": len(overlap),
        "added_count": len(added),
        "removed_count": len(removed),
        "total_snapshot": len(snapshot_triples),
        "overlap_ratio": len(overlap) / max(1, len(original)),
        "added_triples": added,
        "removed_triples": removed,
    }


In [90]:
def analyze_evolution(subgraph, snapshots):
    stats = []
    for snap in snapshots:
        result = match_subgraph_in_snapshot(subgraph, snap)
        stats.append(result)

    # Compute deltas
    overlaps = [s["overlap_ratio"] for s in stats]
    additions = [s["added_count"] for s in stats]
    removals = [s["removed_count"] for s in stats]

    def trend_score(values):
        return values[-1] - values[0]  # simple delta

    delta_overlap = trend_score(overlaps)
    delta_add = trend_score(additions)
    delta_rem = trend_score(removals)

    # Classification logic (simple heuristics)
    if delta_add > 2 and delta_overlap > 0.5:
        label = "Growing"
    elif delta_rem > delta_add and overlaps[-1] < 0.4:
        label = "Decaying"
    elif max(additions) > 2 and sum(additions) / len(additions) > 1.5:
        label = "Getting stronger"
    elif abs(delta_overlap) < 0.1 and abs(delta_add) < 1:
        label = "Stable"
    else:
        label = "Mixed"

    return {
        "evolution": label,
        "overlaps": overlaps,
        "additions": additions,
        "removals": removals
    }


In [91]:
def track_subgraph_evolution_over_time(subgraphs, g, timestamps, t0_index):
    final_outputs = []

    future_ts = timestamps[t0_index+1:]

    snapshots = [extract_snapshot(g, t) for t in future_ts]

    for idx, subgraph in enumerate(subgraphs):
        evolution = analyze_evolution(subgraph, snapshots)
        final_outputs.append({
            "subgraph_id": idx,
            "label": evolution["evolution"],
            "details": evolution,
            "original_subgraph": subgraph
        })

    return final_outputs


In [94]:
index = 1777
evolved_subgraphs = track_subgraph_evolution_over_time(
    subgraphs,
    g,
    timestamps,
    index
)

for s in evolved_subgraphs:
    print(f"Subgraph {s['subgraph_id']}: {s['label']}")


Subgraph 0: Decaying
Subgraph 1: Decaying
Subgraph 2: Decaying
Subgraph 3: Decaying
Subgraph 4: Decaying
Subgraph 5: Decaying
Subgraph 6: Decaying
Subgraph 7: Decaying
Subgraph 8: Decaying
Subgraph 9: Decaying
Subgraph 10: Decaying
Subgraph 11: Decaying


In [102]:
from pyvis.network import Network
import networkx as nx

def triples_to_nx_graph(triples):
    G = nx.DiGraph()
    for h, r, t in triples:
        G.add_edge(str(h), str(t), label=str(r))
    return G

def visualize_subgraph_evolution_pyvis(full_triples, evolved_subgraphs, filename_prefix="evolution_subgraph"):
    full_G = triples_to_nx_graph(full_triples)

    color_map = {
        "Growing": "green",
        "Decaying": "red",
        "Stable": "blue",
        "Getting stronger": "orange",
        "Mixed": "gray"
    }

    for sub in evolved_subgraphs:
        net = Network(notebook=False, directed=True, height="700px", width="100%")
        subgraph = sub["original_subgraph"]
        label = sub["label"]
        sub_id = sub["subgraph_id"]

        sub_G = triples_to_nx_graph(subgraph)
        sub_color = color_map.get(label, "gray")

        # Add all nodes from full graph in light gray
        for node in full_G.nodes():
            net.add_node(node, label=node.split("/")[-1], color="lightgray")

        # Add full edges in gray
        for u, v, d in full_G.edges(data=True):
            net.add_edge(u, v, label=d['label'].split("/")[-1], color="lightgray")

        # Overlay subgraph nodes and edges in color
        for node in sub_G.nodes():
            net.add_node(node, label=node.split("/")[-1], color=sub_color)
        for u, v, d in sub_G.edges(data=True):
            net.add_edge(u, v, label=d['label'].split("/")[-1], color=sub_color, width=3)

        # Save and show
        filename = f"{filename_prefix}_{sub_id}_{label}.html"
        net.save_graph(filename)
        print(f"Saved: {filename}")


In [103]:
# Extract snapshot at time T₀ (for full graph context)
snapshot = extract_snapshot(g, timestamps[index])
full_triples = rdf_to_triples(snapshot)

# Visualize each subgraph evolution in PyVis
visualize_subgraph_evolution_pyvis(full_triples, evolved_subgraphs)

Saved: evolution_subgraph_0_Decaying.html
Saved: evolution_subgraph_1_Decaying.html
Saved: evolution_subgraph_2_Decaying.html
Saved: evolution_subgraph_3_Decaying.html
Saved: evolution_subgraph_4_Decaying.html
Saved: evolution_subgraph_5_Decaying.html
Saved: evolution_subgraph_6_Decaying.html
Saved: evolution_subgraph_7_Decaying.html
Saved: evolution_subgraph_8_Decaying.html
Saved: evolution_subgraph_9_Decaying.html
Saved: evolution_subgraph_10_Decaying.html
Saved: evolution_subgraph_11_Decaying.html


In [18]:
!pip install pyvis networkx


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 756.0/756.0 kB 28.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.6/1.6 MB 48.1 MB/s eta 0:00:00


In [27]:
import networkx as nx
from pyvis.network import Network

# Converts RDF-style triples to NetworkX graph
def triples_to_nx_graph(triples):
    G = nx.DiGraph()
    for h, r, t in triples:
        G.add_edge(str(h), str(t), label=str(r))
    return G

# Visualize with PyVis
def visualize_interactive_subgraphs(full_triples, subgraphs):
    G = triples_to_nx_graph(full_triples)
    net = Network(notebook=True, directed=True, height="600px", width="100%")

    # Add full graph in light gray
    for node in G.nodes():
        #net.add_node(node, label=node.split('/')[-1], color="lightgray")
        # Node label
        net.add_node(node, label=prefixed_label(node), color="lightgray")

    for u, v, d in G.edges(data=True):
        # net.add_edge(u, v, label=d['label'], color="lightgray")
        # Edge label
        net.add_edge(u, v, label=prefixed_label(d['label']), color="lightgray")

    # Add subgraphs with distinct colors
    sub_colors = ["red", "blue", "green", "orange", "purple"]
    for i, subgraph in enumerate(subgraphs):
        sgG = triples_to_nx_graph(subgraph)
        color = sub_colors[i % len(sub_colors)]
        for node in sgG.nodes():
            # net.add_node(node, label=node.split('/')[-1], color=color)
            # Subgraph node
            net.add_node(node, label=prefixed_label(node), color=color)

        for u, v, d in sgG.edges(data=True):
            net.add_edge(u, v, label=d['label'], color=color)
            # Subgraph edge
            net.add_edge(u, v, label=prefixed_label(d['label']), color=color)


    #net.show_buttons(filter_=['physics'])  # Optional: interactive controls
    #net.show("tkg_subgraphs.html")  # Saves and opens in browser
    net.save_graph("tkg_subgraphs.html")

# Example usage
visualize_interactive_subgraphs(triples, subgraphs)
